[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/VaR_CEE_FM/blob/main/VaR_CEE_ConformalFM/VaR_CEE_ConformalFM.ipynb)

# VaR_CEE_ConformalFM

Applies rolling conformal calibration on top of raw foundation model VaR forecasts.
For each date, computes conformal scores from the calibration window and adjusts the
VaR forecast by the empirical quantile of these scores. Also recalibrates ES using
tail returns below the adjusted VaR.

**Published in:** *Can Foundation Models Manage Risk? Zero-Shot VaR and ES Forecasting for CEE Markets*

**Author:** Daniel Traian Pele

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

%matplotlib inline

# ── Configuration ──────────────────────────────────────────
MARKETS = {
    "Romania":  {"index": "BET",   "fx": "EURRON"},
    "Poland":   {"index": "WIG20", "fx": "EURPLN"},
    "Czechia":  {"index": "PX",    "fx": "EURCZK"},
    "Hungary":  {"index": "BUX",   "fx": "EURHUF"},
    "Bulgaria": {"index": "SOFIX", "fx": "USDBGN"},
}
VAR_ALPHAS = [0.01, 0.025, 0.05]
ES_ALPHA = 0.025
ROLLING_WINDOW = 250
FM_MODELS = ["Chronos-2", "TimesFM-2.5", "Moirai-2.0"]

def get_all_series():
    series = []
    for country, info in MARKETS.items():
        series.append((country, info["index"], f"{info['index']}_ret", "index"))
        series.append((country, info["fx"], f"{info['fx']}_ret", "fx"))
    return series

np.random.seed(42)

VAR_DIR = Path("var_forecasts")
OUT_DIR = Path("var_forecasts")
VAR_DIR.mkdir(exist_ok=True)

## Upload Foundation Model Forecast CSVs

Upload the raw FM forecast CSV files (e.g., `Chronos-2_BET_ret.csv`, `TimesFM-2.5_BET_ret.csv`, `Moirai-2.0_BET_ret.csv`, etc.)

Each CSV should have columns: `date, y_true, var_01pct, var_025pct, var_05pct, es_2p5pct`

In [ ]:
from google.colab import files
print("Upload the foundation model forecast CSVs:")
uploaded = files.upload()
# Move to var_forecasts/
for fname, content in uploaded.items():
    dest = VAR_DIR / fname
    with open(dest, "wb") as f:
        f.write(content)
    print(f"  Saved {dest}")
print(f"\nUploaded {len(uploaded)} files")

In [ ]:
def conformal_calibrate(df_raw, window=ROLLING_WINDOW):
    """Apply rolling conformal calibration to raw FM VaR forecasts."""
    df = df_raw.copy()
    n = len(df)

    alpha_cols = {}
    for alpha in VAR_ALPHAS:
        col = f"var_{str(alpha).replace('0.', '')}pct"
        if col in df.columns:
            alpha_cols[alpha] = col

    for alpha, col in alpha_cols.items():
        raw_vals = df[col].values.copy()
        y_vals = df["y_true"].values
        adjusted = raw_vals.copy()

        for t in range(window, n):
            scores = y_vals[t - window:t] - raw_vals[t - window:t]
            q_corr = np.quantile(scores, alpha)
            adjusted[t] = raw_vals[t] + q_corr

        df[col] = adjusted

    if "es_2p5pct" in df.columns and ES_ALPHA in alpha_cols:
        var_col = alpha_cols[ES_ALPHA]
        y_vals = df["y_true"].values
        var_vals = df[var_col].values
        es_vals = df["es_2p5pct"].values.copy()

        for t in range(window, n):
            past_y = y_vals[t - window:t]
            past_var = var_vals[t - window:t]
            tail = past_y[past_y < past_var]
            if len(tail) >= 3:
                es_vals[t] = np.mean(tail)

        df["es_2p5pct"] = es_vals

    df = df.iloc[window:].reset_index(drop=True)
    return df

In [ ]:
print("=" * 60)
print("CONFORMAL CALIBRATION OF FOUNDATION MODELS")
print("=" * 60)

all_series = get_all_series()

for model in FM_MODELS:
    print(f"\n--- {model} + Conformal ---")
    conf_model = f"{model}-Conf"

    for country, raw_id, series_id, stype in all_series:
        raw_file = VAR_DIR / f"{model}_{series_id}.csv"
        out_file = OUT_DIR / f"{conf_model}_{series_id}.csv"

        if not raw_file.exists():
            print(f"  SKIP {series_id}: no raw forecasts")
            continue

        if out_file.exists():
            print(f"  {series_id}: already done")
            continue

        df_raw = pd.read_csv(raw_file)
        df_adj = conformal_calibrate(df_raw)

        df_adj.to_csv(out_file, index=False, float_format="%.8f")
        print(f"  {series_id}: {len(df_raw)} -> {len(df_adj)} rows "
              f"(after {ROLLING_WINDOW}-day calibration)")

print("\n=== Conformal calibration complete ===")

In [ ]:
all_series = get_all_series()
plotted = False

for model in FM_MODELS:
    if plotted:
        break
    for country, raw_id, series_id, stype in all_series:
        if stype != "index":
            continue
        raw_file = VAR_DIR / f"{model}_{series_id}.csv"
        conf_file = VAR_DIR / f"{model}-Conf_{series_id}.csv"
        if raw_file.exists() and conf_file.exists():
            df_raw = pd.read_csv(raw_file, parse_dates=["date"])
            df_conf = pd.read_csv(conf_file, parse_dates=["date"])

            var_col = [c for c in df_raw.columns if "var_01" in c]
            if not var_col:
                var_col = [c for c in df_raw.columns if "var_" in c]
            if var_col:
                var_col = var_col[0]
                fig, ax = plt.subplots(figsize=(14, 4))
                ax.plot(df_raw["date"], df_raw["y_true"], linewidth=0.3, color="black", alpha=0.7, label="Returns")
                ax.plot(df_raw["date"], df_raw[var_col], linewidth=0.5, color="coral", linestyle="--", label=f"{model} raw VaR 1%")
                if var_col in df_conf.columns:
                    ax.plot(df_conf["date"], df_conf[var_col], linewidth=0.5, color="#2ecc71", label=f"{model}-Conf VaR 1%")
                ax.set_title(f"Conformal Calibration: {model} \u2014 {series_id}")
                ax.set_ylabel("Log returns")
                ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.25), ncol=3, frameon=False)
                ax.axhline(y=0, color="gray", linewidth=0.3)
                plt.tight_layout()
                plt.show()
                plotted = True
                break

In [ ]:
import zipfile, io

zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(OUT_DIR.glob("*-Conf_*.csv")):
        zf.write(f, f.name)
zip_buffer.seek(0)
with open("ConformalFM_results.zip", "wb") as fout:
    fout.write(zip_buffer.read())
files.download("ConformalFM_results.zip")
print("Download started: ConformalFM_results.zip")